In [1]:
import pandas as pd
from pathlib import Path
import numpy as np
import os
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

import plotly.express as px
import plotly.graph_objects as go

tqdm.pandas()

For this notebook to run, you should have the file:
`UCSD_AllSites_Merge_PostProcessedSession_20210504_20240930.csv` in the same folder.

The post processed garage level files will be stored in `./UCSD_garage_datasets_PF`

# Read UCSD PowerFlex data

In [ ]:
def read_ucsd_data() -> pd.DataFrame:
    df_ucsd_pf = pd.read_csv(
        Path("UCSD_AllSites_Merge_PostProcessedSession_20210504_20240930.csv"),
        # usecols=[1, 2, 10, 11, 12, 27],
    )
    df_ucsd_pf["Interval average demand kW"] = (
        df_ucsd_pf["Interval average demand kW"]
        .replace(["No data available", "-"], np.nan)
        .astype(float)
    )
    df_ucsd_pf["Interval kWh"] = df_ucsd_pf["Interval kWh"].astype(float)

    # for the missing values in the "Interval average demand kW" column,
    # we will fill them with the kWh values divided by the interval length in hours

    df_ucsd_pf["Interval average demand kW"] = df_ucsd_pf[
        "Interval average demand kW"
    ].fillna(df_ucsd_pf["Interval kWh"] / 0.25)
    return df_ucsd_pf


df_ucsd_pf = read_ucsd_data()

In [ ]:
df_ucsd_pf.columns

## Check for consistent interval length

In [ ]:
df_ucsd_pf_copy = df_ucsd_pf.copy()
df_ucsd_pf_copy = df_ucsd_pf_copy[
    ["Interval start", "Interval end", "Interval average demand kW", "Site"]
]
df_ucsd_pf_copy["Interval start"] = pd.to_datetime(df_ucsd_pf_copy["Interval start"])
df_ucsd_pf_copy["Interval end"] = pd.to_datetime(
    df_ucsd_pf_copy["Interval end"], format="mixed"
)

In [ ]:
# Check that the interval length (between interval start and end) is always the same (15 minutes)
df_ucsd_pf_copy["Interval length"] = (
    df_ucsd_pf_copy["Interval end"] - df_ucsd_pf_copy["Interval start"]
)
df_ucsd_pf_copy[df_ucsd_pf_copy["Interval length"] != pd.Timedelta("15 minutes")]
# We can see that it's when we have the daylight saving time change. therefore we can ignore those.
# This confirms that the interval length is always 15 minutes

## Clean the data

In [ ]:
def clean_ucsd_data(df_ucsd: pd.DataFrame) -> pd.DataFrame:
    df_cleaned = df_ucsd.copy()

    # Keep only the necessary columns
    df_cleaned = df_cleaned[
        [
            "Session start",
            "Session end",
            "Session duration (minutes)",
            "kWh delivered",
            "10-digit UID",
            "Interval start",
            "Interval average demand kW",
            "Site",
            "EVSEID (PFID)",
        ]
    ]

    print(f"Initial number of sessions: {df_cleaned["10-digit UID"].nunique()}")

    # 1. Clean the session duration
    df_cleaned["Session start Ts"] = pd.to_datetime(
        df_cleaned["Session start"], format="mixed"
    )
    df_cleaned["Session end Ts"] = pd.to_datetime(
        df_cleaned["Session end"], format="mixed"
    )
    df_cleaned["Computed duration (minutes)"] = (
        df_cleaned["Session end Ts"] - df_cleaned["Session start Ts"]
    ).dt.total_seconds() / 60
    df_cleaned["Duration discrepancy"] = (
        df_cleaned["Computed duration (minutes)"]
        - df_cleaned["Session duration (minutes)"]
    ).round()
    # Replace duration by computed duration in sessions with a significant energy delivery
    # if difference is around 1 hour (60 minutes), it is because the session is during a daylight
    # saving time change. We can ignore those because our duration discrepancy doesn't take into account
    # the daylight saving time change
    mask_duration_discrepancy = (
        (
            (df_cleaned["Duration discrepancy"] > 2)
            | (df_cleaned["Duration discrepancy"] < -2)
        )
        & (
            df_cleaned["Duration discrepancy"].isin([59, 60, 61, -59, -60, -61])
            == False
        )
        & (df_cleaned["kWh delivered"] >= 1)
    )
    df_cleaned.loc[mask_duration_discrepancy, "Session duration (minutes)"] = (
        df_cleaned.loc[mask_duration_discrepancy, "Computed duration (minutes)"]
    )
    # Update duration discrepancy
    df_cleaned["Duration discrepancy"] = (
        df_cleaned["Computed duration (minutes)"]
        - df_cleaned["Session duration (minutes)"]
    )

    # 2. Remove the sessions with a low energy
    df_cleaned = df_cleaned.loc[(df_cleaned["kWh delivered"] >= 1)]
    print(
        f"Number of sessions after cleaning low energy: {df_cleaned["10-digit UID"].nunique()}"
    )

    # 3. Remove the sessions with duration lower than 15 minutes
    df_cleaned = df_cleaned.loc[(df_cleaned["Session duration (minutes)"] >= 15)]
    print(
        f"Number of sessions after cleaning low duration: {df_cleaned["10-digit UID"].nunique()}"
    )

    # Drop unnecessary columns
    df_cleaned = df_cleaned.drop(
        columns=[
            "Session start Ts",
            "Session end Ts",
            "Computed duration (minutes)",
            "Duration discrepancy",
        ]
    )
    # TODO
    # - Clean "Energy Needed" column. When it is equal to 0 or when "Energy Unit" is not kWh,
    #     we can replace the energy needed by "Miles Needed" * "Wh per mile" / 1000
    # - Clean the cars. Some of them are missing, but we can probably put an average value for the
    #     missing battery capacity and Wh per mile information
    return df_cleaned


df_cleaned = clean_ucsd_data(df_ucsd_pf)

### Optional: Analyze the sessions with a high duration

[Optional] Next we look at the sessions with a very long duration. Some of these sessions are actually multiple sessions grouped together. The following function can be applied to the df to compute how many actual sessions there are in each session of the df.

One way to clean that would be to break down these grouped sessions into separate sessions, but it is not absolutely necessary.

Some of the long sessions are just sessions that didn't stop, but where users have a reasonable amount of energy.

In [55]:
def apply_count_actual_number_of_sessions(group: pd.DataFrame) -> float:
    # Each group is supposed to have only one session. But if we look at the intervals,
    # some groups have gaps in the intervals (e.g. we have a few intervals on a day and then
    # a few on another day a week later, etc.). Such gaps indicate that there are multiple sessions.
    # We will count the number of actual sessions in each group
    try:
        if group.shape[0] == 1:
            return 1
        group.loc[:, "Interval start"] = pd.to_datetime(group["Interval start"])
        interval_length = group["Interval start"].diff().dt.total_seconds() / 60
        gaps = interval_length.value_counts()
        number_of_large_gaps = gaps[
            (abs(gaps.index) > 15) & ((abs(gaps.index) < 59) | (abs(gaps.index) > 61))
        ].sum()
    except Exception as e:
        print(group)
        raise e

    return number_of_large_gaps + 1


# test on a group that should have 1 session
group = df_cleaned.loc[df_cleaned["10-digit UID"] == 520353]
assert apply_count_actual_number_of_sessions(group) == 1

# test on a group that should have 2 sessions
group = df_cleaned.loc[df_cleaned["10-digit UID"] == 378008440]
assert (
    apply_count_actual_number_of_sessions(group) == 25
), f"Found {apply_count_actual_number_of_sessions(group)} actual sessions"

In [ ]:
df_actual_session_number = df_cleaned.groupby("10-digit UID").progress_apply(
    apply_count_actual_number_of_sessions
)

In [ ]:
df_actual_session_number[df_actual_session_number > 1]

### Analyse EVSEs

Here we check if there are multiple sessions on the same EVSE at the same time

In [57]:
df_evses = (
    df_cleaned[["EVSEID (PFID)", "Interval start", "Session ID"]]
    .groupby(["EVSEID (PFID)", "Interval start"])
    .count()
)

The value_counts below shows us that sometimes there is 2 simultaneous sessions on the same EVSE. However, after looking more closely at the data. it appears to be times when a session starts right after the end of the previous session. (e.g. a session end at 10:25, another starts at 10:29 -> it is in the same interval, but it is two distinct sessions)

In [ ]:
df_evses["Session ID"].value_counts()

In [ ]:
test = df_cleaned[
    (df_cleaned["Interval start"] == "2023-02-17 10:15:00")
    & (df_cleaned["EVSEID (PFID)"] == "0051-01-01-01")
]
test

## Read main file

In [ ]:
df_ucsd_pf_copy = read_ucsd_data()
df_ucsd_pf_copy = clean_ucsd_data(df_ucsd_pf_copy)
df_ucsd_pf_copy = df_ucsd_pf_copy[
    ["Interval start", "Interval average demand kW", "Site", "EVSEID (PFID)"]
]

# Rename the Athena garage because it has two names
df_ucsd_pf_copy["Site"] = df_ucsd_pf_copy["Site"].apply(
    lambda x: "UCSD - Athena Garage" if x == "Athena Garage" else x
)

df_ucsd_pf_copy["Interval start"] = pd.to_datetime(df_ucsd_pf_copy["Interval start"])
df_ucsd_pf_copy = df_ucsd_pf_copy.rename(
    columns={"Interval average demand kW": "power", "Interval start": "date"}
)

# convert power from kW to W
df_ucsd_pf_copy["power"] = df_ucsd_pf_copy["power"] * 1000

### Compute number of working EVSEs

In [5]:
df_evses_status = (
    df_ucsd_pf_copy.groupby(["date", "EVSEID (PFID)"])
    .count()["power"]
    .unstack()
    .resample("15min")
    .sum()
    > 0
)

TODO: 
- Chargers should be working for at least 30 min or 1 hour to be considered as working
- Ask for Ryan's definition


In [13]:
# To estimate the number of evses available at each timestep, we use the following methodology:
# - We look at the EVSEs' status between d and d-look_back_days of the current timestep
# - Then on that period, we count the number of evses that were active at least once
# - This gives us the number of available evses at the current timestep

lookback_days = 5

df_number_eveses_available_rolling_7D = (
    df_evses_status.rolling(lookback_days * 24 * 4).sum() > 0
).sum(axis=1)

In [14]:
# We can compare with this simple method that counts the number of evses available
# each week
df_number_eveses_available_7D = (df_evses_status.resample("7D").sum() > 0).sum(axis=1)

In [17]:
# Add the evses availability to df_ucsd_pf_copy
df_ucsd_pf_copy = df_ucsd_pf_copy.merge(
    df_number_eveses_available_rolling_7D.reset_index().rename(
        columns={0: "number_of_evses_available"}
    ),
    on="date",
    how="left",
)

In [ ]:
df_ucsd_pf_copy

In [ ]:
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=df_number_eveses_available_7D.index,
        y=df_number_eveses_available_7D,
        mode="lines",
        name="7D average availability",
    )
)
fig.add_trace(
    go.Scatter(
        x=df_number_eveses_available_rolling_7D.index,
        y=df_number_eveses_available_rolling_7D,
        mode="lines",
        name=f"Past {lookback_days}D rolling availability",
    )
)
average_power = (
    df_ucsd_pf_copy[["date", "power"]].set_index("date").resample("1D").sum().fillna(0)
)
# scale power
average_power = (
    average_power
    * max(df_number_eveses_available_7D)
    / (max(average_power["power"]) * 0.8)
)
fig.add_trace(
    go.Scatter(
        x=average_power.index,
        y=average_power["power"],
        mode="lines",
        name="Power",
    )
)
fig.update_layout(
    title="Number of EVSEs available over time",
    xaxis_title="Date",
    yaxis_title="Number of EVSEs available",
)

In [ ]:
average_power

## Check for missing data

Here are two functions to find and interpolate missing data

In [67]:
def get_df_missing_intervals(df: pd.DataFrame) -> pd.DataFrame:
    # find all intervalls with missing data
    df_missing_intervals = df.dropna(subset=["power"]).copy()

    # find the difference between two data points
    df_missing_intervals["diff"] = df_missing_intervals["date"].diff()
    # dates are missing if the difference is not 5 minutes
    df_missing_intervals = df_missing_intervals[
        df_missing_intervals["diff"] > pd.Timedelta("15min")
    ]

    # define start and end dates of missing intervals
    df_missing_intervals["start_na"] = (
        df_missing_intervals["date"]
        - df_missing_intervals["diff"]
        + pd.Timedelta("15min")
    )
    df_missing_intervals["end_na"] = df_missing_intervals["date"] - pd.Timedelta(
        "15min"
    )
    return df_missing_intervals

In [68]:
from pandas.tseries.holiday import USFederalHolidayCalendar as us_calendar


def interpolate_missing_data(
    power_df_like: pd.DataFrame,
    max_intervaL_length_to_interpolate: pd.Timedelta = pd.Timedelta("5h"),
) -> pd.DataFrame:
    """Function to interpolate missing data
    The function performs two different interpolations when it makes sense:
        1. For missing intervals of less than max_intervaL_length_to_interpolate,
            it interpolates the first half of the interval with the previous value
            (before the start of the interval) and the second half with the next
            value (after the end of the interval).
        2. For missing intervals of less than 24 hours and during the night
            or over a weekend/holidays, it interpolates the missing interval at 0.
    """
    power_df_total_power_interpolated = power_df_like.copy()

    df_missing_intervals = get_df_missing_intervals(power_df_total_power_interpolated)
    number_of_missing_intervals = df_missing_intervals.shape[0]

    if number_of_missing_intervals == 0:
        print("No missing intervals to interpolate.")
        return power_df_like

    cal = us_calendar()
    holidays = cal.holidays(
        start=power_df_like["date"].iloc[0], end=power_df_like["date"].iloc[-1]
    )

    buffer = pd.Timedelta("15min")
    print(
        f"Start of interpolation, there are {number_of_missing_intervals} "
        f"intervals to interpolate."
    )
    interpolated_intervals = 0
    interpolated_intervals_at_0 = 0  # count the intervals that are longer and
    # therefore interpolated at 0 (if it is a weekend or during the night,
    # we can assume that there is no usage)

    for i, missing_interval in tqdm(
        df_missing_intervals.iterrows(), total=number_of_missing_intervals
    ):
        if missing_interval["diff"] < max_intervaL_length_to_interpolate:

            first_half_mask = power_df_total_power_interpolated["date"].between(
                missing_interval["start_na"] - buffer,
                missing_interval["end_na"] - missing_interval["diff"] // 2,
            )
            power_df_total_power_interpolated.loc[first_half_mask] = (
                power_df_total_power_interpolated.loc[first_half_mask].ffill()
            )

            second_half_mask = power_df_total_power_interpolated["date"].between(
                missing_interval["end_na"] - missing_interval["diff"] // 2,
                missing_interval["end_na"] + buffer,
            )
            power_df_total_power_interpolated.loc[second_half_mask] = (
                power_df_total_power_interpolated.loc[second_half_mask].bfill()
            )
            interpolated_intervals += 1
        elif (
            (  # interval during the night
                (
                    missing_interval["start_na"].hour >= 16
                    or missing_interval["start_na"].hour <= 4
                )
                and (missing_interval["end_na"].hour < 12)
            )
            or (  # interval during weekend
                (missing_interval["start_na"].weekday() >= 5)
                or (missing_interval["end_na"].weekday() >= 5)
                or missing_interval["start_na"].date() in (holidays.date)
                or missing_interval["end_na"].date() in (holidays.date)
            )
        ) and (  # we only interpolate night or weekend missing
            # intervals that are less than 24 hours
            missing_interval["diff"]
            < pd.Timedelta("24h")
        ):
            mask = power_df_total_power_interpolated["date"].between(
                missing_interval["start_na"], missing_interval["end_na"]
            )
            power_df_total_power_interpolated.loc[mask, "power"] = (
                power_df_total_power_interpolated.loc[mask, "power"].fillna(0)
            )

            interpolated_intervals += 1
            interpolated_intervals_at_0 += 1

    print(
        f"End of interpolation, {interpolated_intervals} intervals of less than "
        f"{max_intervaL_length_to_interpolate} "
        f"were interpolated ({interpolated_intervals/number_of_missing_intervals*100:.2f}%). "
        f"This includes {interpolated_intervals_at_0} intervals interpolated at 0 "
        "(night or weekends)"
    )
    return power_df_total_power_interpolated

## Function to clean group

In [70]:
def clean_group(df: pd.DataFrame):
    df = df.reset_index()

    df = df.sort_values("date")

    df = df.set_index("date").asfreq("15min").reset_index()
    df = interpolate_missing_data(df)
    df = df.dropna()  # We drop what can't be interpolated

    # Add a column for workday boolean
    df["workday"] = (df["date"].dt.dayofweek < 5).astype(int)
    return df

In [ ]:
df_ucsd_pf_copy[
    df_ucsd_pf_copy["Site"].isin(["UCSD - 8980 La Jolla Drive", "UCSD - Athena Garage"])
]

In [ ]:
test = (
    df_ucsd_pf_copy[
        df_ucsd_pf_copy["Site"].isin(
            ["UCSD - 8980 La Jolla Drive", "UCSD - Athena Garage"]
        )
    ]
    .groupby(["date"])
    .sum()
    .drop(columns=["Site"])
)
display(test)


test = clean_group(test)


test

## Make dataset for each garage

In [72]:
garage_datasets_folder = Path("UCSD_garage_datasets_PF")

In [ ]:
os.makedirs(garage_datasets_folder, exist_ok=True)

df_ucsd_grouped = df_ucsd_pf_copy.groupby(["Site", "date"]).sum()

for garage_name in df_ucsd_grouped.index.get_level_values(0).unique():
    df_garage = df_ucsd_grouped.loc[garage_name]

    df_garage = clean_group(df_garage)
    # Save
    df_garage.to_csv(
        garage_datasets_folder
        / f"{garage_name.replace(' ', '_').replace(":", "")}.csv",
        index=False,
    )
    print(f"{garage_name} has {df_garage.shape[0]:_} rows")
df_garage

## Make aggregated dataset

In [ ]:
df_ucsd_all = df_ucsd_pf_copy.drop(["Site"], axis=1).groupby("date").sum()
df_ucsd_all = clean_group(df_ucsd_all)
df_ucsd_all.to_csv(garage_datasets_folder / "All_Garages.csv", index=False)

# Visualization

In [60]:
start_date = pd.Timestamp("2018-01-01")
end_date = pd.Timestamp("2024-10-01")
number_of_time_ticks = 12
df = df_ucsd_all  # clean_group(
#     df_ucsd_pf_copy.groupby(["Site", "date"]).sum().loc["UCSD Campus Point East"]
# )  #

In [ ]:
# create the subframe
sub_power_df = df.query(f"date >= '{start_date}' and date <= '{end_date}'").copy()
sub_power_df = sub_power_df.set_index("date").asfreq("15min").reset_index()
sub_power_df["isArtificialData"] = sub_power_df["power"].isna()
cal = us_calendar()
holidays = cal.holidays(start=start_date, end=end_date)
sub_power_df["isHoliday"] = sub_power_df["date"].dt.date.isin(holidays.date)
sub_power_df = interpolate_missing_data(sub_power_df)

# Plot the graph

fig, ax1 = plt.subplots(figsize=(10, 3))
# ax2 = ax1.twinx()
ax1.plot(sub_power_df["date"], sub_power_df["power"], color="blue")
ax1.set_ylabel("Power (W)", color="blue")
ymax = sub_power_df["power"].max() * 1.1
ax1.set_ylim(0, ymax)
# add background color for weekends and public holidays
ax1.bar(
    sub_power_df["date"],
    ymax * (abs(sub_power_df["workday"] - 1) + sub_power_df["isHoliday"]),
    width=pd.Timedelta("15Min"),
    align="edge",
    color="lightgrey",
    alpha=0.5,
    label="Weekend or Holiday",
)
# add background color for missing data
ax1.bar(
    sub_power_df["date"],
    ymax * sub_power_df["isArtificialData"],
    width=pd.Timedelta("15Min"),
    align="edge",
    color="red",
    alpha=0.3,
    label="Missing Data",
)


# if mode == "active_sessions":
#     ax2.plot(
#         sub_power_df["date"],
#         sub_power_df["numberOfActiveSessions"],
#         color="green",
#         alpha=0.7,
#     )
#     ax2.set_ylabel("Number of Active Sessions", color="green")
#     ax2.set_ylim(0, 10)
# elif mode == "price":
#     ax2.plot(
#         sub_power_df["date"],
#         sub_power_df["averageRegularPricePerHour"],
#         color="purple",
#         label="Average Regular Price",
#     )
#     ax2.plot(
#         sub_power_df["date"],
#         sub_power_df["reg_centsPerHr"].ffill(),
#         color="pink",
#         alpha=0.7,
#         label="Regular Price from sessions",
#     )
#     ax2.plot(
#         sub_power_df["date"],
#         sub_power_df["averageScheduledPricePerHour"],
#         color="green",
#         label="Average Scheduled Price",
#     )
#     ax2.plot(
#         sub_power_df["date"],
#         sub_power_df["sch_centsPerHr"].ffill(),
#         color="lightgreen",
#         alpha=0.7,
#         label="Scheduled Price from sessions",
#     )

#     ax2.set_ylabel("Price per hour", color="green")
#     ax2.legend(bbox_to_anchor=(0.3, -0.35), loc="upper left", borderaxespad=0)

# Customize the plot

plt.title("Power Readings Over Time")


# set the time ticks
desired_frequencies = [2, 4, 6, 12, 24, 48, 96, 168, 336, 720, 1440]
x_axis_freq = int(
    (end_date - start_date).total_seconds() / 3600 // number_of_time_ticks
)
# Select the desired frequency that is closest to but not greater than the initial frequency
x_axis_freq = max([freq for freq in desired_frequencies if freq <= x_axis_freq * 1.3])
print(
    f"Using x-axis frequency of {x_axis_freq} hours in order to display around {number_of_time_ticks} ticks."
)
x_axis_freq = f"{x_axis_freq}h"
ax1.set_xlim(start_date, end_date)

ax1.set_xticks(pd.date_range(start_date, end_date, freq=x_axis_freq))

ax1.set_xticklabels(
    pd.date_range(start_date, end_date, freq=x_axis_freq).strftime("%y-%m-%d %Hh"),
    rotation=75,
)
ax1.set_xlabel("Timestamp")

ax1.legend(bbox_to_anchor=(0, -0.35), loc="upper left", borderaxespad=0)
plt.grid(True)
fig.subplots_adjust(bottom=0.1)

# Show the plot

plt.show()

Visualize daily energy

In [ ]:
df_ucsd_pf_copy

In [ ]:
(
    df_ucsd_pf_copy.groupby(["Site", "date"])
    .sum()
    .loc[["UCSD - Athena Garage", "UCSD Gilman Parking Structure"]]
)

In [ ]:
start_date = pd.Timestamp("2022-01-01")
end_date = pd.Timestamp("2023-01-01")
number_of_time_ticks = 12
df = clean_group(
    df_ucsd_pf_copy[
        df_ucsd_pf_copy["Site"].isin(
            ["UCSD - Athena Garage", "UCSD Gilman Parking Structure"]
        )
    ]
    .groupby(["date"])
    .sum()
    .drop(columns=["Site"])
)  # df_ucsd_all

In [ ]:
# create the subframe
sub_power_df = df.query(f"date >= '{start_date}' and date <= '{end_date}'").copy()
sub_power_df = sub_power_df.set_index("date").asfreq("15min").reset_index()
sub_power_df["isArtificialData"] = sub_power_df["power"].isna()
cal = us_calendar()
holidays = cal.holidays(start=start_date, end=end_date)
sub_power_df["isHoliday"] = sub_power_df["date"].dt.date.isin(holidays.date)
sub_power_df = interpolate_missing_data(sub_power_df)
sub_power_df = sub_power_df.set_index("date").resample("D").sum().reset_index()
sub_power_df["energy"] = sub_power_df["power"] / 4
# Plot the graph

fig, ax1 = plt.subplots(figsize=(10, 3))
# ax2 = ax1.twinx()
ax1.plot(sub_power_df["date"], sub_power_df["energy"] / 1000, color="blue")
ax1.set_ylabel("Energy (kWh)", color="blue")
ymax = sub_power_df["energy"].max() * 1.1 / 1000
ax1.set_ylim(0, ymax)
# add background color for weekends and public holidays
ax1.bar(
    sub_power_df["date"],
    ymax * (abs(sub_power_df["workday"] - 1) + sub_power_df["isHoliday"]),
    width=pd.Timedelta("15Min"),
    align="edge",
    color="lightgrey",
    alpha=0.5,
    label="Weekend or Holiday",
)
# add background color for missing data
ax1.bar(
    sub_power_df["date"],
    ymax * sub_power_df["isArtificialData"],
    width=pd.Timedelta("15Min"),
    align="edge",
    color="red",
    alpha=0.3,
    label="Missing Data",
)


# Customize the plot

plt.title("Daily Energy Demand Over Time")


# set the time ticks
desired_frequencies = [2, 4, 6, 12, 24, 48, 96, 168, 336, 720, 1440]
x_axis_freq = int(
    (end_date - start_date).total_seconds() / 3600 // number_of_time_ticks
)
# Select the desired frequency that is closest to but not greater than the initial frequency
x_axis_freq = max([freq for freq in desired_frequencies if freq <= x_axis_freq * 1.3])
print(
    f"Using x-axis frequency of {x_axis_freq} hours in order to display around {number_of_time_ticks} ticks."
)
x_axis_freq = f"{x_axis_freq}h"
ax1.set_xlim(start_date, end_date)

ax1.set_xticks(pd.date_range(start_date, end_date, freq=x_axis_freq))

ax1.set_xticklabels(
    pd.date_range(start_date, end_date, freq=x_axis_freq).strftime("%y-%m-%d %Hh"),
    rotation=75,
)
ax1.set_xlabel("Timestamp")

# ax1.legend(bbox_to_anchor=(0, -0.35), loc="upper left", borderaxespad=0)
plt.grid(True)
fig.subplots_adjust(bottom=0.1)

# Show the plot

plt.show()

In [ ]:
sub_power_df["energy"].sum() / 1000

In [ ]:
550 * 360 * 0.15

In [ ]:
2370 * 12